In [ ]:
!pip install openai datasets pandas

In [ ]:
from google.colab import userdata
import os

api_key_openai = userdata.get('OPENAI_API_KEY')

In [ ]:
import pandas as pd
from datasets import load_dataset, Audio, Dataset
from openai import OpenAI

client = OpenAI(api_key=api_key_openai)

def translate_with_llm(text):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "Ти професійний перекладач. Тобі дають уривки з англійських книг. Твоя задача — перекласти їх українською мовою. Уникай дослівного перекладу (кальок), роби текст максимально природним, літературним та зрозумілим. Виводь ЛИШЕ перекладений текст."
                },
                {
                    "role": "user",
                    "content": text
                }
            ],
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Помилка API: {e}")
        return text

def create_llm_ast_dataset(num_samples=30):
    print("1. Завантажуємо аудіо...")
    dummy_data = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

    data_list = []

    print(f"2. Переклад {num_samples} записів через OpenAI API...")
    for i in range(min(num_samples, len(dummy_data))):
        item = dummy_data[i]
        en_text = item['text']

        uk_text = translate_with_llm(en_text)

        data_list.append({
            "audio": item['audio'],
            "en_text": en_text,
            "uk_text": uk_text
        })

        if (i + 1) % 5 == 0:
            print(f"   Готово {i + 1}/{num_samples}...")

    print("3. Формуємо Hugging Face Dataset...")
    ast_dataset = Dataset.from_list(data_list)
    ast_dataset = ast_dataset.cast_column("audio", Audio(sampling_rate=16000))

    return ast_dataset

In [ ]:
dataset_llm = create_llm_ast_dataset(num_samples=30)

df = pd.DataFrame({
    'Оригінал (EN)': dataset_llm['en_text'][:3],
    'LLM Переклад (UK)': dataset_llm['uk_text'][:3]
})
pd.set_option('display.max_colwidth', None)
print("\n--- Якість LLM перекладу ---")
print(df)

1. Завантажуємо аудіо...


README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet:   0%|          | 0.00/9.19M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

2. Переклад 30 записів через OpenAI API...
   Готово 5/30...
   Готово 10/30...
   Готово 15/30...
   Готово 20/30...
   Готово 25/30...
   Готово 30/30...
3. Формуємо Hugging Face Dataset...

--- Якість LLM перекладу ---
                                                                                                                                                                  Оригінал (EN)  \
0                                                                                     MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL   
1                                                                                                               NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER   
2  HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND   

                                                             

In [ ]:
import pandas as pd

df_export = pd.DataFrame({
    'en_text': dataset_llm['en_text'],
    'uk_text': dataset_llm['uk_text']
})

csv_filename = "ast_translations.csv"
df_export.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"Тексти успішно збережено у файл '{csv_filename}'")

dataset_folder = "my_ast_dataset_saved"
dataset_llm.save_to_disk(dataset_folder)
print(f"Повний датасет надійно збережено у папку '{dataset_folder}'")

Тексти успішно збережено у файл 'ast_translations.csv'


Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

Повний датасет надійно збережено у папку 'my_ast_dataset_saved'


In [ ]:
!pip install transformers evaluate jiwer pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 72.6 MB/s eta 0:00:00


In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="uk", task="transcribe")

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

class WhisperDataModule(pl.LightningDataModule):
    def __init__(self, dataset, processor, data_collator, batch_size=4):
        super().__init__()
        self.dataset = dataset
        self.processor = processor
        self.data_collator = data_collator
        self.batch_size = batch_size

    def prepare_dataset(self, batch):
        audio = batch["audio"]
        batch["input_features"] = self.processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

        batch["labels"] = self.processor.tokenizer(batch["uk_text"]).input_ids
        return batch

    def setup(self, stage=None):
        processed_dataset = self.dataset.map(self.prepare_dataset, remove_columns=self.dataset.column_names)
        split_dataset = processed_dataset.train_test_split(test_size=0.2, seed=42)
        self.train_dataset = split_dataset["train"]
        self.val_dataset = split_dataset["test"]

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, collate_fn=self.data_collator, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, collate_fn=self.data_collator)

data_module = WhisperDataModule(dataset=dataset_llm, processor=processor, data_collator=data_collator, batch_size=4)

data_module.setup()

print(f"Кількість батчів для тренування: {len(data_module.train_dataloader())}")
print(f"Кількість батчів для валідації: {len(data_module.val_dataloader())}")

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Кількість батчів для тренування: 6
Кількість батчів для валідації: 2


In [ ]:
import torch
import pytorch_lightning as pl
from transformers import WhisperForConditionalGeneration

class WhisperASTModule(pl.LightningModule):
    def __init__(self, model_name="openai/whisper-tiny", learning_rate=1e-5):
        super().__init__()
        self.save_hyperparameters()

        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)

        self.model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="uk", task="transcribe")
        self.model.config.suppress_tokens = []

    def forward(self, input_features, labels=None):
        return self.model(input_features=input_features, labels=labels)

    def training_step(self, batch, batch_idx):
        input_features = batch["input_features"]
        labels = batch["labels"]

        outputs = self(input_features, labels)
        loss = outputs.loss

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_features = batch["input_features"]
        labels = batch["labels"]

        outputs = self(input_features, labels)
        loss = outputs.loss

        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.hparams.learning_rate)
        return optimizer

ast_model = WhisperASTModule()

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [ ]:
import pytorch_lightning as pl

trainer = pl.Trainer(
    max_epochs=15,
    accelerator="auto",
    devices=1,
    log_every_n_steps=1,
    enable_checkpointing=False
)

print("Запуск тренування...")
trainer.fit(model=ast_model, datamodule=data_module)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Запуск тренування...


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                            ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ WhisperForConditionalGeneration │ 37.8 M │ eval │     0 │
└───┴───────┴─────────────────────────────────┴────────┴──────┴───────┘

Trainable params: 37.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 37.8 M                                                                                               
Total estimated model params size (MB): 151                                                                        
Modules in train mode: 0                                                                                           
Modules in eval mode: 126                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 126 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=15` reached.


In [ ]:
import torch

ast_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
ast_model.to(device)

ast_model.model.config.suppress_tokens = None
ast_model.model.config.forced_decoder_ids = None

sample = dataset_llm[-2]
audio = sample["audio"]

print("Оригінальний текст (EN):", sample["en_text"])
print("Очікуваний переклад (UK):", sample["uk_text"])

input_features = processor.feature_extractor(
    audio["array"],
    sampling_rate=audio["sampling_rate"],
    return_tensors="pt"
).input_features.to(device)

print("\nМодель слухає та перекладає...")

forced_ids = processor.get_decoder_prompt_ids(language="uk", task="transcribe")

with torch.no_grad():
    predicted_ids = ast_model.model.generate(
        input_features,
        forced_decoder_ids=forced_ids,
        suppress_tokens=[]
    )

predicted_text = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("Результат нашої моделі:", predicted_text)

Оригінальний текст (EN): INQUIRED SHAGGY IN THE METAL FOREST
Очікуваний переклад (UK): ЗАПИТУВАВ ШЕРШАВИЙ У МЕТАЛЕВОМУ ЛІСІ

Модель слухає та перекладає...
Результат нашої моделі:  Ін kwиод шегі, і на медл ворост.
